# Chapter 0 — The Gaia DR3 non-single-star catalogue

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/gaia-nss-tutorial/blob/main/notebooks/00_data.ipynb)

**Goal.** Gaia DR3 (June 2022) was the first Gaia release with a catalogue of binary-star solutions: about 800 000
orbits, accelerations and trends, from astrometry, spectroscopy and photometry (Gaia Collaboration, Arenou et al. 2023,
https://ui.adsabs.harvard.edu/abs/2023A&A...674A..34G). This chapter introduces the four non-single-star (NSS) tables,
the data files of this tutorial, and how the three channels — astrometric, spectroscopic, eclipsing — see different
binaries.

**You will learn**
1. what the NSS tables contain, and the difference between a *solution*, a *source* and a physical system;
2. how to download the tutorial's data files and check them;
3. where the binaries of each channel sit in the colour–magnitude diagram and in period;
4. that the astrometric solutions were computed only for stars preselected by their RUWE: the catalogue is a selection,
   not a census.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts, attached to a GitHub release of the tutorial. MANIFEST.json lists, for each file,
# its origin (SQL query or URL) and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/gaia-nss-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)

## 0.1 Where the data come from

Students do not need database access. The files were extracted from the Gaia DR3 tables in WSDB (the Whole Sky
Database in Cambridge) by `scripts/build_data.py`; the same tables are in the Gaia archive
(https://gea.esac.esa.int/archive/). For each file, the manifest records the query that made it:

In [ ]:
for name in ('nss_two_body', 'nss_acceleration', 'nss_other'):
    m = MANIFEST[name]
    print(f"{name}: {m['rows']:,} rows, {m['bytes'] / 1e6:.0f} MB\n   {m['queries'][0][:300]} ...\n")

Each NSS row is joined to its star's row in `gaia_source`, so we also have the photometry, the single-star astrometry
and the binarity indicators of Chapters 2 and 4. Note that a source in an NSS table has *two* astrometric solutions:
the single-star one in `gaia_source` (columns `parallax`, `pmra`, …, and the RUWE that describes its misfit), and the
NSS one (columns `nss_parallax`, `nss_pmra`, …). For binaries the NSS parallax is the better one.

## 0.2 The four tables

| Table | Solution types | What is measured |
|---|---|---|
| `nss_two_body_orbit` | `Orbital`, `OrbitalAlternative(Validated)`, `OrbitalTargetedSearch(Validated)` | astrometric orbit of the photocentre |
| | `SB1`, `SB2`, `SB1C`, `SB2C` | spectroscopic orbit of one or both components (C: circular) |
| | `AstroSpectroSB1`, `EclipsingSpectro` | combined solutions |
| | `EclipsingBinary` | orbit from the eclipse light curve |
| `nss_acceleration_astro` | `Acceleration7`, `Acceleration9` | constant or changing acceleration of the photocentre |
| `nss_non_linear_spectro` | `FirstDegreeTrendSB1`, `SecondDegreeTrendSB1` | linear or quadratic trend of the radial velocity |
| `nss_vim_fl` | `VIMF` | variability-induced motion: a variable star in an unresolved pair |

In [ ]:
tbo, acc, oth = load('nss_two_body'), load('nss_acceleration'), load('nss_other')
counts = pd.concat([tbo.nss_solution_type, acc.nss_solution_type, oth.nss_solution_type]).value_counts()
print(counts.to_string())
n_sol = len(tbo) + len(acc) + len(oth)
n_src = pd.concat([tbo.source_id, acc.source_id, oth.source_id]).nunique()
print(f'\n{n_sol:,} solutions for {n_src:,} distinct sources')
# reproducibility: Gaia Collaboration, Arenou et al. (2023), Table 1 and Sect. 2
pub = {'Acceleration7': 246947, 'Acceleration9': 91268, 'Orbital': 134598, 'AstroSpectroSB1': 33467,
       'EclipsingSpectro': 155, 'EclipsingBinary': 86918, 'FirstDegreeTrendSB1': 24083, 'SecondDegreeTrendSB1': 32725,
       'VIMF': 870}
assert all(counts[k] == v for k, v in pub.items())
assert counts[['SB1', 'SB2', 'SB1C', 'SB2C']].sum() == 186905
assert counts[['OrbitalAlternative', 'OrbitalAlternativeValidated']].sum() == 629
assert counts[['OrbitalTargetedSearch', 'OrbitalTargetedSearchValidated']].sum() == 533
assert (n_sol, n_src) == (839098, 813687)

Every count agrees with Table 1 of Arenou et al. (2023). There are 25 411 more solutions than sources: 25 403 sources
have two solutions. Almost all of these pair an astrometric solution with a spectroscopic one (`Orbital` + `SB1`:
5290; an acceleration with an `SB1` orbit or an RV trend: 18 830), because the channels were processed
independently and combined only where the combined fit succeeded (`AstroSpectroSB1`).

The column `non_single_star` of `gaia_source` summarises this: bit 0 (value 1) is set if the source has an
astrometric solution, bit 1 (value 2) a spectroscopic one, bit 2 (value 4) an eclipsing one. Gaia BH1 has value 1, Gaia
BH2 value 3 (astrometric and spectroscopic: an `AstroSpectroSB1` orbit).

In [ ]:
# sources of the two-body and acceleration tables (nss_other does not carry the column)
allsrc = pd.concat([tbo[['source_id', 'non_single_star']], acc[['source_id', 'non_single_star']]]).drop_duplicates('source_id')
bits = pd.DataFrame({'astrometric (1)': allsrc.non_single_star & 1 > 0, 'spectroscopic (2)': allsrc.non_single_star & 2 > 0,
                     'eclipsing (4)': allsrc.non_single_star & 4 > 0})
print(bits.sum().to_string())
print('Gaia BH1, BH2:', tbo.set_index('source_id').loc[[4373465352415301632, 5870569352746779008], ['nss_solution_type', 'non_single_star']].to_string())

## 0.3 Each channel sees different binaries

Figure 0.1 places the binaries of each channel in the colour–magnitude diagram: absolute magnitude
$M_G = G + 5\log_{10}(\varpi/{\rm mas}) - 10$ against $G_{\rm BP} - G_{\rm RP}$, for sources with parallax S/N above 5
(not corrected for extinction). For astrometric solutions we use the NSS parallax, for the others the single-star one
(as Arenou et al. 2023, their Fig. 4).

In [ ]:
def absmag(d, nss=False):
    plx = d.nss_parallax if nss else d.parallax
    plx_err = d.nss_parallax_error if nss else d.parallax_error
    ok = plx / plx_err > 5
    return d.bp_rp[ok], (d.phot_g_mean_mag + 5 * np.log10(plx) - 10)[ok]


groups = {'astrometric orbits (Orbital)': (tbo[tbo.nss_solution_type == 'Orbital'], True),
          'accelerations': (acc, True),
          'spectroscopic orbits (SB1, SB2)': (tbo[tbo.nss_solution_type.isin(['SB1', 'SB2', 'SB1C', 'SB2C'])], False),
          'eclipsing binaries': (tbo[tbo.nss_solution_type == 'EclipsingBinary'], False)}
fig, ax = plt.subplots(1, 4, figsize=(18, 5), sharex=True, sharey=True)
for a, (title, (d, nss)) in zip(ax, groups.items()):
    c, M = absmag(d, nss)
    a.hist2d(c, M, bins=150, range=[[-0.5, 4], [-5, 15]], norm=LogNorm(), cmap='viridis')
    a.set(title=f'{title}\n{len(c):,} with $\\varpi/\\sigma_\\varpi > 5$', xlabel='$G_{\\rm BP} - G_{\\rm RP}$')
ax[0].set_ylabel('$M_G$'); ax[0].invert_yaxis()
plt.tight_layout(); plt.show()

**Figure 0.1.** Colour–magnitude diagrams (absolute $G$ magnitude against $G_{\rm BP} - G_{\rm RP}$, log-scaled
counts) of the sources with astrometric orbits, accelerations, spectroscopic orbits and eclipsing-binary orbits in
Gaia DR3, for parallax S/N > 5 (NSS parallax for the first two).

The channels sample different stars. Astrometric orbits and accelerations (left) follow the main sequence from F to
late-M dwarfs, with few giants: a giant is luminous, hence typically distant, and its photocentre wobble in mas is
small. A thin white-dwarf sequence at $M_G > 10$ appears among the orbits: white dwarfs with low-mass companions.
Every astrometric solution has parallax S/N > 5 (the titles count all of them); this is a consequence of the
publication filter on parallax (Chapter 3), not of the cut in the plot. Spectroscopic orbits (third panel) require
$G_{\rm RVS} \lesssim 12$, a bright limit that favours luminous stars: red giants and the red clump at $M_G \approx 0.5$
are prominent. Eclipsing binaries (right) are mostly main-sequence stars; only 45 929 of 86 918 have a parallax S/N
above 5, because most are faint and distant.

## 0.4 Periods

The channels are sensitive to different periods (Figure 0.2). Eclipses need the stars to be close, so eclipsing
binaries have periods of days. Spectroscopic orbits need velocity changes larger than the Gaia RVS errors
(~1 km/s for bright stars), within the 34 months of data. Astrometric orbits need the photocentre to move by more
than ~0.1 mas, which favours wide (long-period) orbits of nearby stars, up to about the length of the mission.

In [ ]:
bins = np.logspace(-1, 4, 101)
for t, lab in [('EclipsingBinary', 'eclipsing'), (['SB1', 'SB2', 'SB1C', 'SB2C'], 'spectroscopic'),
               ('Orbital', 'astrometric'), ('AstroSpectroSB1', 'astrometric + spectroscopic')]:
    s = tbo[tbo.nss_solution_type.isin(np.atleast_1d(t))]
    plt.hist(s.period, bins, histtype='step', lw=2, label=f'{lab} (median {s.period.median():.0f} d)')
plt.axvline(1038, color='k', ls=':', lw=1, label='DR3 baseline (1038 d)')
plt.axvline(365.25, color='0.5', ls='--', lw=1, label='1 yr')
plt.xscale('log'); plt.yscale('log'); plt.xlabel('period [days]'); plt.ylabel('solutions per bin'); plt.legend(fontsize=8)
plt.show()
med = {k: tbo.period[tbo.nss_solution_type.isin(v)].median() for k, v in
       {'EB': ['EclipsingBinary'], 'SB': ['SB1', 'SB2', 'SB1C', 'SB2C'], 'astro': ['Orbital']}.items()}
# descriptive: median periods are ordered eclipsing < spectroscopic < astrometric
assert med['EB'] < med['SB'] < med['astro']

**Figure 0.2.** Period distributions of the two-body solutions by channel (100 logarithmic bins from 0.1 to
$10^4$ days). Dotted: the DR3 baseline; dashed: one year.

The three distributions barely overlap. Eclipsing binaries peak at 0.3–0.5 d (contact binaries) with a median of 1 d.
Spectroscopic orbits span 0.2 d to about 1500 d, roughly flat in $\log P$ between 10 and 1000 d, with a median of
117 d. Astrometric orbits start at ~30 d (shorter orbits are too small to see) and peak just below the baseline
(median 600 d); the dip at exactly one year is visible in the green histogram (Chapter 3 explains it). All three
channels end at about the 34-month baseline: an orbit must be (nearly) closed within the data to be fitted.

## 0.5 The astrometric solutions are a selection

The NSS astrometric pipeline did not try to fit every star. It processed only sources whose single-star solution was
poor, RUWE > 1.4, brighter than $G \approx 19$, with enough observations (Halbwachs et al. 2023,
https://ui.adsabs.harvard.edu/abs/2023A&A...674A...9H), and published only solutions that passed strict filters
(Chapter 3). The exception is `OrbitalTargetedSearch`, where a list of candidates from other surveys was fitted
whatever their RUWE.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for t in ['Orbital', 'AstroSpectroSB1', 'OrbitalTargetedSearch']:
    s = tbo[tbo.nss_solution_type == t]
    ax[0].hist(s.ruwe, np.logspace(-0.1, 2, 100), histtype='step', lw=2, density=True, label=t)
    ax[1].hist(s.phot_g_mean_mag, np.linspace(3, 21, 91), histtype='step', lw=2, density=True)
for t in ['Acceleration7', 'Acceleration9']:
    s = acc[acc.nss_solution_type == t]
    ax[0].hist(s.ruwe, np.logspace(-0.1, 2, 100), histtype='step', lw=2, density=True, label=t)
    ax[1].hist(s.phot_g_mean_mag, np.linspace(3, 21, 91), histtype='step', lw=2, density=True)
ax[0].axvline(1.4, color='k', ls=':'); ax[0].set(xscale='log', xlabel='RUWE (single-star solution)', ylabel='density')
ax[0].legend(fontsize=8); ax[1].set(xlabel='G [mag]')
plt.tight_layout(); plt.show()
astro = pd.concat([tbo[tbo.nss_solution_type.isin(['Orbital', 'AstroSpectroSB1'])], acc])
print(f'astrometric solutions (Orbital, AstroSpectroSB1, Acceleration7/9): min RUWE = {astro.ruwe.min():.4f}, '
      f'max G = {astro.phot_g_mean_mag.max():.2f}; OrbitalTargetedSearch min RUWE = '
      f"{tbo.ruwe[tbo.nss_solution_type == 'OrbitalTargetedSearch'].min():.2f}")
# the preselection: every main-pipeline astrometric solution has RUWE >= 1.4
assert astro.ruwe.min() >= 1.4

**Figure 0.3.** Distributions of the single-star RUWE (left, dotted line at 1.4) and of the G magnitude (right) of
the stars with astrometric solutions, by solution type.

The RUWE distribution of every main-pipeline solution type starts abruptly at 1.4: none of these stars had a good
single-star fit. Only the targeted search (green), run on input lists of known or suspected binaries and exoplanet
hosts, reaches below 1.4; those stars are bright ($G \approx 3$–10). `AstroSpectroSB1` solutions (orange) are brighter
than $G \approx 13$, the limit of RVS epoch velocities; the purely astrometric ones peak at $G \approx 13.5$ and stop
at $G \approx 18.6$. Any statistic computed from NSS astrometric binaries — a binary fraction, a period distribution —
describes binaries that were selected by RUWE first. Chapter 1 showed which binaries that selection keeps.

## Summary

- Gaia DR3 contains 839 098 NSS solutions for 813 687 sources in four tables: two-body orbits (astrometric,
  spectroscopic, eclipsing and combined), astrometric accelerations, spectroscopic trends and variability-induced movers.
- The three channels see different binaries: eclipsing ones at periods of days, spectroscopic ones at days to years,
  astrometric ones at months to years, and in different parts of the colour–magnitude diagram.
- Astrometric solutions exist only for stars with RUWE > 1.4 (except the targeted search): any statistic of NSS
  astrometric binaries inherits this selection.

**Exercises.** (1) Which sources have both an `EclipsingBinary` and an `SB1` solution? Compare their periods.
(2) Plot Figure 0.1 for `AstroSpectroSB1` solutions. (3) What fraction of `Orbital` solutions have periods within 10%
of one year? Compare with neighbouring period intervals of the same width.